In [1]:
import pandas as pd
import plotly.express as px

In [11]:
df = pd.read_csv("data/standings_data.csv")
df = df[df.manager.notnull()]
len(df)

1150

In [12]:
import numpy as np

In [13]:
tmp = df.copy()
tmp['streak_type'] = tmp['streak'].str[0]
tmp2 = tmp[['season', 'week', 'manager', 'streak_type']].copy()
tmp2['week'] = tmp2['week'] - 1
tmp2.rename(columns={'streak_type': 'next_streak_type'}, inplace=True)
tmp = tmp.merge(tmp2, on=['season', 'week', 'manager'], how='left')
tmp['max_streak'] = np.where(tmp['streak_type'] != tmp['next_streak_type'], tmp['streak'], None)
tmp['max_streak'] = np.where(tmp['max_streak'].isin(['L1', 'W1']), None, tmp['max_streak'])
tmp = tmp[tmp['max_streak'].notna()]
tmp['streak_length'] = tmp['max_streak'].str[1:].astype(int)

streak_counts = tmp.groupby(['streak_type', 'streak_length']).size().reset_index().rename(columns={0: 'number'})
streak_counts.sort_values('number', inplace=True)
fig = px.funnel(streak_counts, x='streak_length', y='number', color='streak_type',
                    color_discrete_sequence=px.colors.qualitative.G10,
                    )

In [14]:
tmp

,season,week,manager,wins,losses,points_for,points_against,standings_rank,leader_wins,games_back,playoff_spot,streak,streak_type,next_streak_type,max_streak,streak_length
11,2021,2,Chris,2.0,0.0,965.35,806.83,2,2.0,0.0,1,W2,W,L,W2,2
18,2021,2,Garth,0.0,2.0,852.58,894.00,9,2.0,2.0,0,L2,L,W,L2,2
19,2021,2,Andy,0.0,2.0,727.33,957.50,10,2.0,2.0,0,L2,L,W,L2,2
22,2021,3,Rich,2.0,1.0,1337.76,1295.43,3,3.0,1.0,1,W2,W,L,W2,2
24,2021,3,Dave,1.0,2.0,1333.49,1332.58,5,3.0,2.0,0,L2,L,W,L2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1131,2026,5,Andy,4.0,1.0,1799.15,1753.31,2,4.0,0.0,1,W2,W,L,W2,2
1142,2026,6,Sam,4.0,2.0,2229.92,2164.85,3,4.0,0.0,1,W3,W,NaN,W3,3
1143,2026,6,Rouse,4.0,2.0,2205.58,2133.01,4,4.0,0.0,1,W2,W,NaN,W2,2
1148,2026,6,Chris,1.0,5.0,1996.51,2286.42,9,4.0,3.0,0,L4,L,NaN,L4,4


In [15]:
fig.show()

In [2]:
df = pd.read_csv("data/team_week_points.csv")

In [3]:
SEASON = 2026
WEEK = df[df['season'] == SEASON]['week'].max()
WEEK_TYPE = df[(df['season'] == SEASON) & (df['week'] == WEEK)]['week_type'].iloc[0]
tmp = df[df['week_type'] == WEEK_TYPE]


In [4]:
from utils import stat_summary
    
hit = pd.read_csv("data/hitting_stats.csv")
pitch = pd.read_csv("data/pitching_stats.csv")
stat_summary(hit, SEASON, WEEK)
stat_summary(pitch, SEASON, WEEK)

,pts,IP,H,ER,BB,K,QS,SV+H,ERA,WHIP,K_per_IP
manager,,,,,,,,,,,
Rouse,186,53,49,13,13,56,6,0,2.17,1.15,1.04
Dave,169,63,56,32,21,58,4,0,4.55,1.22,0.92
Rich,159,55,53,31,17,48,4,3,4.98,1.25,0.86
Sam,155,50,45,20,5,42,4,2,3.58,0.99,0.83
Howard,154,58,65,25,20,51,3,0,3.81,1.44,0.86
Ulmer,153,51,44,23,18,54,3,1,4.01,1.20,1.05
Andy,150,50,57,26,25,49,3,4,4.65,1.63,0.97
Chris,149,54,62,32,15,54,3,2,5.24,1.40,0.98
Garth,121,39,31,18,12,35,4,1,4.08,1.08,0.88


In [ ]:

# Overlay highlights in red
fig.add_scatter(
    x=tmp.loc[highlight_mask, '_dummy_x'],
    y=tmp.loc[highlight_mask, 'margin_pct'],
    mode='markers',
    marker=dict(color='red', size=12, symbol='star'),
    hovertemplate='Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>Margin %: %{y:.3f}<extra></extra>',
    customdata=tmp.loc[highlight_mask, ['season', 'week']].values,

)
fig.update_xaxes(range=[-0.2, 0.2], showticklabels=False, title_text='')
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
hist_top20 = (player_top
    .query("rank <= 20")
    .groupby(['player_type', 'manager_name', 'season', 'week'])
    .size().reset_index(name='count')
)
for player_type in hist_top20['player_type'].unique():
    print(f"Top 20 {player_type}s by manager:")
    plt.hist(hist_top20[hist_top20['player_type'] == player_type]['count'], bins=range(0, 21), align='left')
    plt.xlim(0, 10)
    plt.show()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

SEASON = 2025
WEEK = 2
tmp = hit.copy()
highlight_mask = (tmp['season'] == SEASON) & (tmp['week'] == WEEK)
tmp['_dummy_x'] = 0

for col in ['R', 'RBI', 'R', 'SB', 'BB', 'HBP', '1B', '2B', '3B', 'HR', 'TB']:
    fig = px.strip(tmp,
                x='_dummy_x',
                y=col,
                color_discrete_sequence=['gray'],
                hover_data={'season': True, 'week': True, col: True, '_dummy_x': False},  # Only show what you want
                )
    fig.data[0].hovertemplate = 'Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>' + col + ': %{y:.3f}<extra></extra>'
    fig.data[0].customdata = tmp[['season', 'week']].values

    # Overlay highlights in red
    fig.add_scatter(
        x=tmp.loc[highlight_mask, '_dummy_x'],
        y=tmp.loc[highlight_mask, col],
        mode='markers',
        marker=dict(color='red', size=12, symbol='star'),
        hovertemplate='Season: %{customdata[0]}<br>Week: %{customdata[1]}<br>' + col + ': %{y:.3f}<extra></extra>',
        customdata=tmp.loc[highlight_mask, ['season', 'week']].values,

    )
    fig.update_xaxes(range=[-0.2, 0.2], showticklabels=False, title_text='')
    fig.update_layout(showlegend=False)
    fig.show()